# Week 3 · Tutorial 1 — What figures are for, bar charts and histograms

**Covers videos 3.1 – 3.2:** choosing a figure · counting things (bar charts) and counting numbers (histograms).

**Datasets**
* `generation_mix.csv` — annual electricity generation (TWh) by source for 14 European countries. **One row is one country.**
* `household_kwh.csv` — annual consumption (kWh) of 5 000 households with their dwelling type (`flat`, `terraced`, `detached`). **One row is one household.**

Each ✏️ Your turn asks you to change one choice in a figure (sorting, axis, number of bins, normalisation) **and** to read a number
off the result. *A figure is an argument: every choice is a claim about what matters.*

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/blasern/data-tutorials/main/data/"   # where the data files live
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

## 1. The choice is nearly automatic

| what you have | what to draw |
|---|---|
| one category | bar chart |
| one number | histogram |
| two numbers | scatter plot |
| a number over time | line plot |
| a number split by category | boxplot |

A figure is for **exploring** (quick, many, for yourself) or **explaining** (one, polished, for someone else).

## 2. Bar charts: sorted by size, not by name

In [ ]:
mix = pd.read_csv(DATA_URL + "generation_mix.csv")
print(mix.shape)
mix

In [ ]:
wind = mix.set_index("country")["wind_twh"]

fig, ax = plt.subplots(figsize=(8, 3.5))
wind.plot(kind="bar", ax=ax)           # alphabetical order -> hard to read
ax.set_ylabel("wind generation (TWh)")
plt.show()

### ✏️ Your turn 1

Sort the bars **by value** (largest first) instead of by name.

1. Fill in the `TODO`: `wind_sorted = wind.sort_values(ascending=False)`, and plot that.
2. The cell prints the three largest wind producers and the share of the total wind output of all 14 countries that comes from these three (3 decimals).
3. Fill in the second `TODO`: `wind_share` = wind output divided by the country's **total generation** (the sum of the six `*_twh` columns), per country. Print the country with the highest wind share and its value (3 decimals).

**What to change** — edit the cell exactly like this, then run it:

1. Change `wind_sorted = wind.copy()` into `wind_sorted = wind.sort_values(ascending=False)`

2. Change `wind_share = pd.Series(np.nan, index=wind.index)` into `wind_share = wind / total`

In [ ]:
wind_sorted = wind.copy()     # TODO: sort by value, largest first

fig, ax = plt.subplots(figsize=(8, 3.5))
wind_sorted.plot(kind="bar", ax=ax)
ax.set_ylabel("wind generation (TWh)")
plt.show()

top3 = list(wind_sorted.index[:3])
print("top 3:", top3)
print("share of total wind output from the top 3:", round(wind_sorted.iloc[:3].sum() / wind_sorted.sum(), 3))

total = mix.set_index("country").filter(like="_twh").sum(axis=1)
wind_share = pd.Series(np.nan, index=wind.index)     # TODO: wind / total
print("highest wind share:", wind_share.idxmax() if wind_share.notna().any() else None,
      round(wind_share.max(), 3) if wind_share.notna().any() else None)

### The y-axis of a bar chart starts at zero

The *length* of a bar encodes the value. If the axis starts above zero, the bars lie about the ratio between values.
Mean annual consumption by dwelling type:

In [ ]:
hh = pd.read_csv(DATA_URL + "household_kwh.csv")
means = hh.groupby("dwelling_type")["annual_kwh"].mean().sort_values()
print(hh.shape)
print(means.round(0))

### ✏️ Your turn 2

Quantify how much a truncated axis exaggerates a difference.

The *visual* ratio of two bars with the axis starting at `ymin` is `(b − ymin) / (a − ymin)`, where `b` is the larger and `a` the smaller mean.

1. Set `ymin = 2000` (starter: 0) and run.
2. The cell prints the true ratio (detached / flat) and the visual ratio on the truncated axis (2 decimals).

**What to change** — edit the cell exactly like this, then run it:

Change `ymin = 0` into `ymin = 2000`

In [ ]:
ymin = 0       # TODO: 2000

a, b = means["flat"], means["detached"]
true_ratio = b / a
visual_ratio = (b - ymin) / (a - ymin)

fig, ax = plt.subplots(figsize=(5, 3))
means.plot(kind="bar", ax=ax)
ax.set_ylim(ymin, None); ax.set_ylabel("mean annual kWh")
plt.show()

print("true ratio detached / flat   :", round(true_ratio, 2))
print("visual ratio with ymin =", ymin, ":", round(visual_ratio, 2))

## 3. Histograms: counting numbers

A histogram divides the range of a numerical variable into **bins** and counts the observations in each. The number of bins is
the one choice that matters: *six bins* hide the shape, *six hundred* show only noise.

In [ ]:
x = hh["annual_kwh"]
print(x.describe().round(0))
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(x, bins=20)
ax.set_xlabel("annual consumption (kWh)"); ax.set_ylabel("households")
plt.show()

### ✏️ Your turn 3

Try three bin counts: **6**, **60** and **600**. Change `n_bins` at the `TODO` and run the cell each time.

The cell prints, for the chosen number of bins: the height of the tallest bin, the number of **non-empty** bins, and the left edge of the tallest bin (nearest kWh).
Also run it once with the automatic rule `bins="auto"` (set `n_bins = "auto"`) and note the number of bins it chooses.

In [ ]:
n_bins = 20       # TODO: 6, then 60, then 600, then "auto"

counts, edges = np.histogram(x, bins=n_bins)
print("number of bins:", len(counts))
print("tallest bin holds:", counts.max(), "households")
print("non-empty bins:", int((counts > 0).sum()))
print("left edge of the tallest bin:", round(edges[counts.argmax()]))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(x, bins=n_bins)
ax.set_xlabel("annual consumption (kWh)"); ax.set_ylabel("households")
plt.show()

### Comparing groups of very different size: counts or densities?

There are 3 200 flats but only 700 detached houses. In a *count* histogram the small group is squeezed flat; the **density** (`density=True`) makes the *area* of each histogram equal to 1, so the shapes can be compared.

### ✏️ Your turn 4

Use common bin edges `edges = np.linspace(0, 30000, 61)` for all groups.

1. Run once with `density = False` and note the **tallest bin of the flats** and of the detached houses.
2. Change to `density = True` and run again: the heights now are densities (per kWh). The tall flat peak and the low detached
   peak both become comparable.
3. The cell also prints the share of each group with annual consumption **above 10 000 kWh** (3 decimals).

**What to change** — edit the cell exactly like this, then run it:

1. Change `density = False` into `fig, ax = plt.subplots(figsize=(7, 3.2))`

2. Replace
```python

fig, ax = plt.subplots(figsize=(7, 3.2))
for name in ["flat", "terraced", "detached"]:
    vals = hh.loc[hh["dwelling_type"] == name, "annual_kwh"]
    h, _ = np.histogram(vals, bins=edges, density=density)
    ax.stairs(h, edges, label=name)
    print(f"{name:9s} n={len(vals):5d}  tallest bin = {h.max():.6f}  share above 10 000 kWh = {(vals > 10000).mean():.3f}")
ax.legend(); ax.set_xlabel("annual consumption (kWh)")
ax.set_ylabel("density" if density else "households")
plt.show()
```
by
```python
for density in [False, True]:
    print("density =", density)
    for name in ["flat", "terraced", "detached"]:
        vals = hh.loc[hh["dwelling_type"] == name, "annual_kwh"]
        h, _ = np.histogram(vals, bins=edges, density=density)
        print(f"  {name:9s} n={len(vals):5d}  tallest bin = {h.max():.6f}  share above 10 000 kWh = {(vals > 10000).mean():.3f}")
```

In [ ]:
density = False      # TODO: True
edges = np.linspace(0, 30000, 61)

fig, ax = plt.subplots(figsize=(7, 3.2))
for name in ["flat", "terraced", "detached"]:
    vals = hh.loc[hh["dwelling_type"] == name, "annual_kwh"]
    h, _ = np.histogram(vals, bins=edges, density=density)
    ax.stairs(h, edges, label=name)
    print(f"{name:9s} n={len(vals):5d}  tallest bin = {h.max():.6f}  share above 10 000 kWh = {(vals > 10000).mean():.3f}")
ax.legend(); ax.set_xlabel("annual consumption (kWh)")
ax.set_ylabel("density" if density else "households")
plt.show()

### ✏️ Your turn 5

A figure is an argument. Write the *claim* a reader should take from this figure as the title — and check it with a number.

Fill in the `TODO`: compute the **median** annual consumption of each dwelling type (nearest kWh) into `medians`, and the ratio `ratio_detached_flat` = median of detached / median of flat (1 decimal).
Then set an informative title in the plot (the starter title is just a description).

**What to change** — edit the cell exactly like this, then run it:

1. Replace
```python
medians = pd.Series(dtype=float)
ratio_detached_flat = np.nan
```
by
```python
medians = hh.groupby("dwelling_type")["annual_kwh"].median()
ratio_detached_flat = medians["detached"] / medians["flat"]
```

2. Change `ax.set_title("Annual consumption by dwelling type")` into `ax.set_title("A detached house uses about 3.8 times as much electricity as a flat")`

In [ ]:
medians = pd.Series(dtype=float)      # TODO: median annual_kwh per dwelling_type
ratio_detached_flat = np.nan          # TODO

if len(medians):
    print(medians.round(0))
    print("median detached / median flat:", round(ratio_detached_flat, 1))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(hh.loc[hh["dwelling_type"] == "flat", "annual_kwh"], bins=edges, density=True, alpha=0.6, label="flat")
ax.hist(hh.loc[hh["dwelling_type"] == "detached", "annual_kwh"], bins=edges, density=True, alpha=0.6, label="detached")
ax.set_title("Annual consumption by dwelling type")      # TODO: replace by the claim
ax.legend(); plt.show()

## Summary

* Pick the figure from the data type: category → bar, number → histogram.
* Sort bars by value (unless the categories have a natural order), start bar axes at zero.
* The number of bins changes the story; use density when group sizes differ.
* Title your figure with the claim and make sure the numbers support it.